# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### My action policy

The model/ranking is used as decision-support for prioritizing content items for human review.

I use three action labels:

- HIGH_PRIORITY: review first because the item has a strong measured opportunity signal.
- REVIEW: review after the high-priority items.
- MONITOR: keep under observation rather than acting immediately.

The reason code explains the main observable signal behind the recommendation. The queue is a prioritization tool, not an automatic publishing or refresh decision.

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Load the starter dataset
data_path = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print("Columns:", list(df.columns))

Shape: (30000, 44)
Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


In [3]:
# ---------------------------------------------------------
# ML-10 ACTION SCORE
# ---------------------------------------------------------

work_df = df.copy()

# Safe numeric conversion
numeric_cols = [
    "search_volume",
    "impressions_90d",
    "clicks_90d",
    "engaged_sessions_90d",
    "days_since_last_update",
    "ctr",
    "avg_position"
]

for col in numeric_cols:
    if col in work_df.columns:
        work_df[col] = pd.to_numeric(work_df[col], errors="coerce")

# Percentile ranks make different scales comparable.
def pct_rank(series):
    return series.rank(pct=True, method="average").fillna(0)

# Signals:
# 1. staleness
# 2. search demand
# 3. impressions/opportunity
work_df["staleness_score"] = pct_rank(
    work_df["days_since_last_update"]
)

work_df["demand_score"] = pct_rank(
    work_df["search_volume"]
)

work_df["impression_score"] = pct_rank(
    work_df["impressions_90d"]
)

# Transparent hand-written score.
work_df["action_score"] = (
    0.50 * work_df["staleness_score"]
    + 0.30 * work_df["demand_score"]
    + 0.20 * work_df["impression_score"]
)

# One reason code per row.
work_df["reason_code"] = np.select(
    [
        work_df["staleness_score"] >= 0.75,
        work_df["demand_score"] >= 0.75,
    ],
    [
        "STALE_CONTENT",
        "HIGH_SEARCH_DEMAND",
    ],
    default="GENERAL_REVIEW"
)

# Action label
work_df["action"] = np.select(
    [
        work_df["action_score"] >= 0.75,
        work_df["action_score"] >= 0.50,
    ],
    [
        "HIGH_PRIORITY",
        "REVIEW",
    ],
    default="MONITOR"
)

# Rank highest score first
work_df = work_df.sort_values(
    "action_score",
    ascending=False
).reset_index(drop=True)

work_df["rank"] = np.arange(1, len(work_df) + 1)

# Show queue
queue_cols = [
    "rank",
    "content_id",
    "client_id",
    "action_score",
    "action",
    "reason_code"
]

queue = work_df[queue_cols].copy()

print("Action distribution:")
print(queue["action"].value_counts())

print("\nTop 20:")
display(queue.head(20))

Action distribution:
action
MONITOR          15103
REVIEW           12373
HIGH_PRIORITY     2524
Name: count, dtype: int64

Top 20:


,rank,content_id,client_id,action_score,action,reason_code
0,1,content_5fe46e04994d,client_4e07408562,0.917459,HIGH_PRIORITY,STALE_CONTENT
1,2,content_e6955a2c59dc,client_4e07408562,0.914171,HIGH_PRIORITY,STALE_CONTENT
2,3,content_b242bb46cb5e,client_3fdba35f04,0.913835,HIGH_PRIORITY,STALE_CONTENT
3,4,content_2e0b3dc70916,client_4e07408562,0.912812,HIGH_PRIORITY,STALE_CONTENT
4,5,content_05b3cf6119c2,client_19581e27de,0.912777,HIGH_PRIORITY,STALE_CONTENT
5,6,content_2725d2bcfac1,client_4e07408562,0.912175,HIGH_PRIORITY,STALE_CONTENT
6,7,content_62abc4bd66be,client_4e07408562,0.911935,HIGH_PRIORITY,STALE_CONTENT
7,8,content_979a999506bd,client_19581e27de,0.911558,HIGH_PRIORITY,STALE_CONTENT
8,9,content_2c2606c5d176,client_19581e27de,0.911180,HIGH_PRIORITY,STALE_CONTENT
9,10,content_90e4f1f70ab4,client_19581e27de,0.911057,HIGH_PRIORITY,STALE_CONTENT


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended to help a human reviewer prioritize which content items should be checked first.

The score is decision-support. It does not automatically recommend publishing, deleting, or changing content.

### Limits

The ranking is based on observed signals in the available dataset. A high score does not prove that refreshing a page will improve performance.

The score can be wrong when:
- the underlying measurements are unusual or incomplete;
- a page is already intentionally stale;
- search demand does not represent a realistic opportunity;
- the content has a business or editorial constraint not represented in the data.

Human review is required before action.

In [4]:
# Basic sanity checks for the action queue

print("Rows ranked:", len(queue))
print("Unique content IDs:", queue["content_id"].nunique())

print("\nScore range:")
print(queue["action_score"].describe())

print("\nActions:")
print(queue["action"].value_counts())

print("\nReason codes:")
print(queue["reason_code"].value_counts())

Rows ranked: 30000
Unique content IDs: 30000

Score range:
count    30000.000000
mean         0.487677
std          0.193700
min          0.017323
25%          0.337484
50%          0.497742
75%          0.647134
max          0.917459
Name: action_score, dtype: float64

Actions:
action
MONITOR          15103
REVIEW           12373
HIGH_PRIORITY     2524
Name: count, dtype: int64

Reason codes:
reason_code
GENERAL_REVIEW        15882
STALE_CONTENT          9091
HIGH_SEARCH_DEMAND     5027
Name: count, dtype: int64


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review checklist

Before taking action, a reviewer should check:

1. The content item is still relevant to the intended search need.
2. The observed search-demand signal is plausible.
3. The page is not already undergoing an editorial or technical change.
4. The apparent staleness is meaningful rather than intentional.
5. There is no known business, legal, brand, or editorial restriction.

### No-go list

The system should never automatically:

- publish content;
- delete content;
- change titles or claims;
- make legal/compliance decisions;
- override editorial decisions;
- treat the score as proof of future traffic improvement.

The model/rule only prioritizes what a human should review.

In [5]:
# Create a human-review checklist for the top recommendations.

top_review = work_df.head(20).copy()

review_cols = [
    "rank",
    "content_id",
    "action",
    "reason_code",
    "action_score",
    "days_since_last_update",
    "search_volume",
    "impressions_90d"
]

available_review_cols = [
    c for c in review_cols if c in top_review.columns
]

display(top_review[available_review_cols])

,rank,content_id,action,reason_code,action_score,days_since_last_update,search_volume,impressions_90d
0,1,content_5fe46e04994d,HIGH_PRIORITY,STALE_CONTENT,0.917459,104,1900.0,517715
1,2,content_e6955a2c59dc,HIGH_PRIORITY,STALE_CONTENT,0.914171,104,3600.0,37534
2,3,content_b242bb46cb5e,HIGH_PRIORITY,STALE_CONTENT,0.913835,104,1600.0,56997
3,4,content_2e0b3dc70916,HIGH_PRIORITY,STALE_CONTENT,0.912812,104,9900.0,27948
4,5,content_05b3cf6119c2,HIGH_PRIORITY,STALE_CONTENT,0.912777,104,3600.0,32019
5,6,content_2725d2bcfac1,HIGH_PRIORITY,STALE_CONTENT,0.912175,104,6600.0,27348
6,7,content_62abc4bd66be,HIGH_PRIORITY,STALE_CONTENT,0.911935,104,2900.0,31364
7,8,content_979a999506bd,HIGH_PRIORITY,STALE_CONTENT,0.911558,104,880.0,67982
8,9,content_2c2606c5d176,HIGH_PRIORITY,STALE_CONTENT,0.911180,104,590.0,347399
9,10,content_90e4f1f70ab4,HIGH_PRIORITY,STALE_CONTENT,0.911057,104,720.0,82372


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

### Monitoring triggers

The recommendations should be reconsidered when:

- the distribution of key input signals changes substantially;
- the share of HIGH_PRIORITY recommendations changes sharply;
- the observed data becomes stale;
- important features have unexpected missingness;
- later validation shows that the ranking quality has degraded.

A retrain/rebuild should be considered when the underlying data distribution or the definition of the business task changes.

The score should not be treated as permanently valid just because it worked on the current dataset.

In [6]:
# Record simple monitoring statistics for this run.

monitoring = pd.DataFrame({
    "metric": [
        "rows",
        "high_priority_share",
        "review_share",
        "monitor_share",
        "median_action_score",
        "median_days_since_last_update",
        "median_search_volume",
        "median_impressions_90d"
    ],
    "value": [
        len(work_df),
        (work_df["action"] == "HIGH_PRIORITY").mean(),
        (work_df["action"] == "REVIEW").mean(),
        (work_df["action"] == "MONITOR").mean(),
        work_df["action_score"].median(),
        work_df["days_since_last_update"].median(),
        work_df["search_volume"].median(),
        work_df["impressions_90d"].median()
    ]
})

display(monitoring)

,metric,value
0,rows,30000.000000
1,high_priority_share,0.084133
2,review_share,0.412433
3,monitor_share,0.503433
4,median_action_score,0.497742
5,median_days_since_last_update,20.000000
6,median_search_volume,10.000000
7,median_impressions_90d,731.000000


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

### Export

The final ranked queue and monitoring statistics are written to `work/outputs/`.

These outputs are receipts for the analysis and can be reused in the final paper/capstone.

In [7]:
# ---------------------------------------------------------
# EXPORTS
# ---------------------------------------------------------

output_dir = REPO_ROOT / "work" / "outputs"
output_dir.mkdir(parents=True, exist_ok=True)

# Ranked action queue
queue_path = output_dir / "ml10_action_queue.csv"
queue.to_csv(queue_path, index=False)

# Monitoring statistics
monitoring_path = output_dir / "ml10_monitoring_metrics.json"
monitoring.to_json(monitoring_path, orient="records", indent=2)

# Also save a compact summary JSON
summary = {
    "rows_ranked": int(len(work_df)),
    "high_priority": int((work_df["action"] == "HIGH_PRIORITY").sum()),
    "review": int((work_df["action"] == "REVIEW").sum()),
    "monitor": int((work_df["action"] == "MONITOR").sum()),
    "median_action_score": float(work_df["action_score"].median())
}

import json

summary_path = output_dir / "ml10_action_summary.json"

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print("Written:")
print(queue_path)
print(monitoring_path)
print(summary_path)

print("\nTop 10 action queue:")
display(queue.head(10))

Written:
/Users/theshivrajpatil/FlyRank-ML-Internship/work/outputs/ml10_action_queue.csv
/Users/theshivrajpatil/FlyRank-ML-Internship/work/outputs/ml10_monitoring_metrics.json
/Users/theshivrajpatil/FlyRank-ML-Internship/work/outputs/ml10_action_summary.json

Top 10 action queue:


,rank,content_id,client_id,action_score,action,reason_code
0,1,content_5fe46e04994d,client_4e07408562,0.917459,HIGH_PRIORITY,STALE_CONTENT
1,2,content_e6955a2c59dc,client_4e07408562,0.914171,HIGH_PRIORITY,STALE_CONTENT
2,3,content_b242bb46cb5e,client_3fdba35f04,0.913835,HIGH_PRIORITY,STALE_CONTENT
3,4,content_2e0b3dc70916,client_4e07408562,0.912812,HIGH_PRIORITY,STALE_CONTENT
4,5,content_05b3cf6119c2,client_19581e27de,0.912777,HIGH_PRIORITY,STALE_CONTENT
5,6,content_2725d2bcfac1,client_4e07408562,0.912175,HIGH_PRIORITY,STALE_CONTENT
6,7,content_62abc4bd66be,client_4e07408562,0.911935,HIGH_PRIORITY,STALE_CONTENT
7,8,content_979a999506bd,client_19581e27de,0.911558,HIGH_PRIORITY,STALE_CONTENT
8,9,content_2c2606c5d176,client_19581e27de,0.911180,HIGH_PRIORITY,STALE_CONTENT
9,10,content_90e4f1f70ab4,client_19581e27de,0.911057,HIGH_PRIORITY,STALE_CONTENT


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.